# True Parametric PINN for full impact period (e.g., 50 impacts)

Train one model on multiple non-parametric PINN trajectories and predict an unseen parameter case over the full horizon.

This notebook also saves weights/training logs and compares unseen prediction vs true non-parametric result.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'parametric_PINN'))
sys.path.insert(0, str(ROOT / 'Dispersion'))

from true_parametric_pinn import (
    ParametricModelConfig,
    TrueParametricPINN,
    build_supervised_dataset,
)
from pinn_dispersion_from_mat import load_pinn_results

In [ ]:
# ---------------- USER SETTINGS ----------------
# Training cases from previous non-parametric PINN outputs
train_mat_files = [
    '../Result/pinn_data_phi1_1_phi2_10.mat',
    '../Result/pinn_data_phi1_1p5_phi2_15.mat',
    '../Result/pinn_data_phi1_2_phi2_20.mat',
    '../Result/pinn_data_phi1_1p2_phi2_18.mat',
]

# Unseen case for validation (must exist from previous non-parametric PINN run)
unseen_mat_file = '../Result/pinn_data_phi1_1p6_phi2_16.mat'

n_iter = 4000
print_every = 400
checkpoint_dir = '../Result/parametric_model_ckpt'
checkpoint_tag = 'true_parametric_full_impact'

In [ ]:
# Load training trajectories and build supervised dataset
t_list, x_list, phi_cases = [], [], []
for fp in train_mat_files:
    t_total, x_total, params = load_pinn_results(fp)
    t_list.append(t_total)
    x_list.append(x_total)
    phi_cases.append((float(params['phi1']), float(params['phi2'])))

X_data, Y_data = build_supervised_dataset(t_list, x_list, phi_cases)

phi1_vals = [p[0] for p in phi_cases]
phi2_vals = [p[1] for p in phi_cases]
t_max = max(float(np.max(t)) for t in t_list)
n_dof = x_list[0].shape[1]

cfg = ParametricModelConfig(
    n_dof=n_dof,
    m_x=1.0,
    k=1.0,
    c=0.0,
    seg_window=t_max,
    phi1_min=min(phi1_vals),
    phi1_max=max(phi1_vals),
    phi2_min=min(phi2_vals),
    phi2_max=max(phi2_vals),
    layers=(3, 64, 64, n_dof),
    beta_data=1.0,
    lr=1e-3,
)

print('X_data shape:', X_data.shape)
print('Y_data shape:', Y_data.shape)
print('phi1 range:', (cfg.phi1_min, cfg.phi1_max))
print('phi2 range:', (cfg.phi2_min, cfg.phi2_max))
print('time range:', (0.0, cfg.seg_window))

In [ ]:
# Train once on full trajectories (full impact horizon)
model = TrueParametricPINN(cfg)
model.train_supervised(X_data, Y_data, n_iter=n_iter, print_every=print_every)

# Save weights + training results
model.save_checkpoint(checkpoint_dir, tag=checkpoint_tag)
print('Saved checkpoint to:', checkpoint_dir)

In [ ]:
# Unseen parameter comparison against true non-parametric result
t_true, x_true, params_true = load_pinn_results(unseen_mat_file)
phi1_u = float(params_true['phi1'])
phi2_u = float(params_true['phi2'])

x_pred, _ = model.predict(t_true.reshape(-1,1), phi1=phi1_u, phi2=phi2_u)

rmse_per_dof = np.sqrt(np.mean((x_pred - x_true)**2, axis=0))
rmse_global = float(np.sqrt(np.mean((x_pred - x_true)**2)))

print(f'Unseen case: phi1={phi1_u}, phi2={phi2_u}')
print('Global RMSE:', rmse_global)
print('RMSE first 5 DOFs:', rmse_per_dof[:5])

In [ ]:
# Plot comparison (last DOF)
plt.figure(figsize=(9, 4))
plt.plot(t_true, x_true[:, -1], 'k-', lw=2, label='True non-parametric PINN (last DOF)')
plt.plot(t_true, x_pred[:, -1], 'r--', lw=2, label='True parametric PINN prediction (last DOF)')
plt.xlabel('t')
plt.ylabel('x_last')
plt.title('Unseen-parameter comparison over full impact horizon')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot training loss log
plt.figure(figsize=(7, 3.5))
plt.semilogy(model.loss_log, lw=1.6)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('True parametric PINN supervised training loss')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()